In [1]:
# Nếu chạy Colab L4, cell này chạy 1 lần là đủ.
!pip -q install pandas numpy scikit-learn tqdm torch transformers accelerate bitsandbytes sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.1 MB/s eta 0:00:00


In [2]:
!pip install -U huggingface_hub hf_transfer
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 124.8 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0


In [3]:
import os
import re
import json
import random
import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from sklearn.metrics import cohen_kappa_score, mean_absolute_error, mean_squared_error

import torch
import torch.nn as nn

from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer


/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:277: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


In [4]:
# Reproducibility
SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cuda'

In [5]:
TRAIN_PATH = "/content/asap_train.csv"
VAL_PATH   = "/content/asap_val.csv"
TEST_PATH  = "/content/asap_test.csv"

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

required_cols = ["essay_id", "essay_set", "essay", "domain1_score", "domain1_score_norm"]

for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    missing = set(required_cols) - set(df.columns)
    assert not missing, f"{name} missing columns: {missing}"

train_df.head()

,essay_id,essay_set,essay,domain1_score,domain1_score_norm
0,14876,6,In the excerpt The Mooring Mast by Marcia Amid...,3.0,0.75
1,9985,4,The author concluded this sentence because he ...,0.0,0.00
2,3231,2,"I can think of several books that, I would not...",4.0,0.60
3,21137,8,My best friend @PERSON2 turned thirteen on a b...,39.0,0.65
4,17919,7,A time that I was patient is when I broke my f...,23.0,0.77


In [6]:
ESSAY_SET = 1

p1_train = train_df[train_df["essay_set"] == ESSAY_SET].copy().reset_index(drop=True)
p1_val   = val_df[val_df["essay_set"] == ESSAY_SET].copy().reset_index(drop=True)
p1_test  = test_df[test_df["essay_set"] == ESSAY_SET].copy().reset_index(drop=True)

print("P1 train:", p1_train.shape)
print("P1 val:  ", p1_val.shape)
print("P1 test: ", p1_test.shape)

p1_train[["essay_id", "essay_set", "domain1_score", "domain1_score_norm"]].head()

P1 train: (1248, 5)
P1 val:   (180, 5)
P1 test:  (355, 5)


,essay_id,essay_set,domain1_score,domain1_score_norm
0,1311,1,10.0,0.8
1,735,1,9.0,0.7
2,1152,1,8.0,0.6
3,1314,1,10.0,0.8
4,1645,1,11.0,0.9


In [7]:
ASAP_SCORE_RANGES = {
    1: (2, 12),
    2: (1, 6),
    3: (0, 3),
    4: (0, 3),
    5: (0, 4),
    6: (0, 4),
    7: (0, 30),
    8: (0, 60),
}

Y_MIN, Y_MAX = ASAP_SCORE_RANGES[ESSAY_SET]
Y_MIN, Y_MAX

(2, 12)

In [8]:
def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)

    qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    spearman = pd.Series(y_true).corr(pd.Series(y_pred), method="spearman")

    return {
        "QWK": float(qwk),
        "MAE": float(mae),
        "RMSE": float(rmse),
        "Spearman": float(spearman) if not pd.isna(spearman) else np.nan,
    }


In [9]:
def sample_pairs(df, num_pairs=1000, seed=42, min_per_essay=3):
    rng = np.random.default_rng(seed)
    essay_ids = df["essay_id"].astype(int).tolist()

    pairs = set()
    counts = {eid: 0 for eid in essay_ids}

    # 1. Ensure each essay appears at least min_per_essay times if possible
    attempts = 0
    max_attempts = num_pairs * 200

    while min(counts.values()) < min_per_essay and len(pairs) < num_pairs and attempts < max_attempts:
        underrepresented = [eid for eid, c in counts.items() if c < min_per_essay]

        a = int(rng.choice(underrepresented))
        b = int(rng.choice([eid for eid in essay_ids if eid != a]))

        x, y = sorted((a, b))

        if (x, y) not in pairs:
            pairs.add((x, y))
            counts[x] += 1
            counts[y] += 1

        attempts += 1

    # 2. Fill remaining pairs randomly
    while len(pairs) < num_pairs:
        a, b = rng.choice(essay_ids, size=2, replace=False)
        x, y = sorted((int(a), int(b)))
        pairs.add((x, y))

    return pd.DataFrame(list(pairs), columns=["essay_id_1", "essay_id_2"])


# Paper-like transductive setup:
# LCES main experiments score the target essay set as a whole. The scoring
# model is trained from comparisons among the essays to be scored, then the
# learned latent scores are linearly mapped to the rubric range.
p1_train_split = p1_train.copy()
p1_val_split = p1_val.copy()
p1_test_split = p1_test.copy()

p1_train_split["split"] = "train"
p1_val_split["split"] = "val"
p1_test_split["split"] = "test"

p1_all = pd.concat(
    [p1_train_split, p1_val_split, p1_test_split],
    axis=0,
    ignore_index=True
)

# Paper uses M=5,000 pairwise comparisons. You can set this to 6000 if you
# want to reuse your previous budget, but 5000 is the paper-like default.
M_PAIRWISE = 6500

all_pairs = sample_pairs(
    p1_all,
    num_pairs=M_PAIRWISE,
    seed=SEED,
    min_per_essay=4
)

print("P1 all:", p1_all.shape)
print("Pairwise comparisons M:", all_pairs.shape)
all_pairs.head()


P1 all: (1783, 6)
Pairwise comparisons M: (6500, 2)


,essay_id_1,essay_id_2
0,1453,1521
1,1255,1765
2,1109,1482
3,191,848
4,1507,1757


In [10]:
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"
# Có thể dùng v0.2 nếu v0.3 lỗi access/package:
# MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

# Rất quan trọng với decoder-only model
tokenizer.padding_side = "left"
tokenizer.truncation_side = "left"

# Mistral thường không có pad_token riêng.
# Cách ổn nhất là dùng eos_token làm pad_token.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

llm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

llm.config.pad_token_id = tokenizer.pad_token_id
llm.generation_config.pad_token_id = tokenizer.pad_token_id
llm.eval()

print("Loaded:", MODEL_ID)
print("padding_side:", tokenizer.padding_side)
print("pad_token:", tokenizer.pad_token)
print("pad_token_id:", tokenizer.pad_token_id)
print("eos_token:", tokenizer.eos_token)
print("eos_token_id:", tokenizer.eos_token_id)
print("model pad_token_id:", llm.config.pad_token_id)
print("generation pad_token_id:", llm.generation_config.pad_token_id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Loaded: mistralai/Mistral-7B-Instruct-v0.3
padding_side: left
pad_token: </s>
pad_token_id: 2
eos_token: </s>
eos_token_id: 2
model pad_token_id: 2
generation pad_token_id: 2


In [11]:
P1_RUBRIC_COMPACT = """
Evaluate the quality of a persuasive letter or essay about the effects computers have on people.

Task:
The student must write a persuasive letter to a local newspaper stating an opinion about the effects computers have on people and persuading readers to agree.

A strong response should take a clear position, support that position with convincing reasons and examples, address the newspaper audience, stay focused on the topic, and be organized and readable.

Score range:
This essay set is scored from 2 to 12. When comparing two essays, judge which one would receive the higher overall score.

High-quality responses:
Prefer the essay that:
- clearly states a position about the effects computers have on people,
- maintains that position throughout the response,
- gives relevant, convincing, and well-developed reasons,
- uses specific examples, observations, or experiences to support the argument,
- explains why the reasons matter instead of merely listing them,
- is focused on the topic and avoids unrelated ideas,
- has clear organization with a beginning, middle, and end,
- uses transitions to connect ideas,
- has an effective introduction and conclusion,
- uses appropriate tone for a letter to a local newspaper,
- shows good vocabulary, sentence variety, and fluency,
- has stronger control of grammar, spelling, capitalization, punctuation, and usage.

Medium-quality responses:
These responses may state an opinion and give some reasons, but the support may be general, repetitive, uneven, or only partly persuasive.
They may have basic organization but weaker transitions, limited development, or some language and convention problems.

Low-quality responses:
These responses may have an unclear or inconsistent position, little relevant support, weak organization, minimal development, frequent errors, or may only loosely address the prompt.

Very low-quality responses:
These responses may be off topic, too brief, confusing, irrelevant, or contain too little information to judge.

When comparing two essays, prioritize:
1. clarity and consistency of the persuasive position,
2. relevance and development of reasons and examples,
3. organization and coherence,
4. audience awareness and persuasive tone,
5. language quality and writing conventions.

Choose "tie" only if the two essays are genuinely very close in overall quality.
""".strip()

In [12]:
P1_PROMPT_TEXT = """
Computers and Society

More and more people use computers, but not everyone agrees that this benefits society.

Those who support advances in technology believe that computers have a positive effect on people.
They argue that computers teach hand-eye coordination, give people the ability to learn about faraway places and people,
and allow people to communicate online with others.

Others have different ideas. Some experts are concerned that people spend too much time on computers and less time
exercising, enjoying nature, and interacting with family and friends.

Writing Task:
Write a persuasive letter to your local newspaper in which you state your opinion on the effects computers have on people.
Persuade the readers to agree with your position.

A strong response should:
- clearly state an opinion about whether computers have positive or negative effects on people,
- support that opinion with convincing reasons and examples,
- address the audience of a local newspaper,
- stay focused on the effects computers have on people,
- be organized like a persuasive letter or essay,
- use clear language and appropriate tone.
""".strip()

In [13]:
SYSTEM_PROMPT_ASAP = """
You are an English teacher evaluating middle school exam essays.
Compare the two essays using the prompt and rubric.
Return only one valid JSON object.
Do not copy the essays.
Do not write anything outside the JSON.
""".strip()


def build_pairwise_prompt(essay1, essay2):
    return f"""
# Task
Which essay would receive a higher overall score?

# Prompt
{P1_PROMPT_TEXT}

# Rubric Guidelines
{P1_RUBRIC_COMPACT}

# Rules
- Do not penalize anonymized entities such as PERSON, ORGANIZATION, LOCATION, DATE, TIME, MONEY, PERCENT, CAPS, or NUM.
- Return only one valid JSON object.
- The value must be exactly one of: "essay1", "essay2", "tie".
- Do not explain.
- Do not copy the essays.

# Essay1
{essay1}

# Essay2
{essay2}

# Answer
Return your decision now as one valid JSON object only.
Use exactly this key: "preference".
Allowed values: "essay1", "essay2", "tie".

JSON:
""".strip()

In [14]:
def extract_preference(text):
    """Extract preference from JSON-like LLM output with robust fallbacks."""
    raw = "" if text is None else str(text)
    text = raw.strip()

    def normalize_pref(pref):
        pref = str(pref).lower().strip()
        pref = pref.replace(" ", "")
        pref = pref.replace("_", "")
        pref = pref.replace('"', "")
        pref = pref.replace("'", "")

        if pref in ["essay1", "1", "first", "firstessay"]:
            return "essay1"
        if pref in ["essay2", "2", "second", "secondessay"]:
            return "essay2"
        if pref in ["tie", "equal", "same", "samequality", "both", "neither"]:
            return "tie"
        return None

    def make_result(reasoning="", preference="tie", parse_failed=False):
        return {
            "reasoning": str(reasoning)[:500],
            "preference": preference,
            "raw_text": raw[:2000],
            "parse_failed": parse_failed,
        }

    # Empty output
    if text == "":
        return make_result("", "tie", parse_failed=True)

    # Remove markdown fences if model returns ```json ... ```
    cleaned = text
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned.strip(), flags=re.IGNORECASE)
    cleaned = re.sub(r"\s*```$", "", cleaned.strip())

    # 1. Direct JSON parse
    try:
        obj = json.loads(cleaned)
        if isinstance(obj, dict):
            pref = normalize_pref(obj.get("preference", ""))
            if pref is not None:
                return make_result(obj.get("reasoning", ""), pref, parse_failed=False)
    except Exception:
        pass

    # 2. Extract JSON-looking blocks and try each one
    json_blocks = re.findall(r"\{[^{}]*\}", cleaned, flags=re.DOTALL)
    for block in json_blocks:
        try:
            obj = json.loads(block)
            if isinstance(obj, dict):
                pref = normalize_pref(obj.get("preference", ""))
                if pref is not None:
                    return make_result(obj.get("reasoning", ""), pref, parse_failed=False)
        except Exception:
            continue

    # 3. Regex preference field from imperfect JSON/text
    pref_match = re.search(
        r'["\']?preference["\']?\s*[:=]\s*["\']?(essay\s*1|essay\s*2|essay1|essay2|tie|equal|same)',
        cleaned,
        flags=re.IGNORECASE,
    )
    if pref_match:
        pref = normalize_pref(pref_match.group(1))
        if pref is not None:
            return make_result(cleaned, pref, parse_failed=False)

    # 4. Final-decision style fallback
    decision_match = re.search(
        r'(decision|final decision|answer|winner)\s*[:=\-]?\s*["\']?(essay\s*1|essay\s*2|essay1|essay2|tie|equal|same)',
        cleaned,
        flags=re.IGNORECASE,
    )
    if decision_match:
        pref = normalize_pref(decision_match.group(2))
        if pref is not None:
            return make_result(cleaned, pref, parse_failed=False)

    # 5. Safer last-resort tail check
    # Only accept if exactly one of essay1 / essay2 appears in the tail.
    tail = cleaned[-500:].lower()

    has_essay1 = re.search(r"\bessay\s*1\b|\bessay1\b", tail) is not None
    has_essay2 = re.search(r"\bessay\s*2\b|\bessay2\b", tail) is not None
    has_tie = re.search(r"\btie\b|same score|equal quality|equally|same quality", tail) is not None

    if has_tie and not has_essay1 and not has_essay2:
        return make_result(cleaned, "tie", parse_failed=False)

    if has_essay1 and not has_essay2:
        return make_result(cleaned, "essay1", parse_failed=False)

    if has_essay2 and not has_essay1:
        return make_result(cleaned, "essay2", parse_failed=False)

    # True parser failure
    return make_result(cleaned, "tie", parse_failed=True)


tokenizer.padding_side = "left"

@torch.inference_mode()
def call_local_llm(prompts, max_new_tokens=32, temperature=0.0):
    """Batched local LLM inference for pairwise judgments."""
    input_texts = []

    tokenizer.padding_side = "left"
    tokenizer.truncation_side = "left"

    for prompt in prompts:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT_ASAP},
            {"role": "user", "content": prompt},
        ]

        if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template is not None:
            input_text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
        else:
            input_text = SYSTEM_PROMPT_ASAP + "\n\n" + prompt

        input_texts.append(input_text)

    inputs = tokenizer(
        input_texts,   # important: use chat-template text, not raw prompts
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=2048,
    ).to(llm.device)

    do_sample = temperature is not None and temperature > 0

    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "use_cache": True,
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }

    if do_sample:
        generation_kwargs["temperature"] = temperature

    with torch.no_grad():
        output_ids = llm.generate(
            **inputs,
            **generation_kwargs,
        )

    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[:, prompt_len:]

    texts = tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True,
    )

    results = [extract_preference(t) for t in texts]

    del inputs, output_ids, generated_ids

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results


In [15]:
# Quick smoke test
sample_essay_1 = p1_all.iloc[0]["essay"]
sample_essay_2 = p1_all.iloc[1]["essay"]

test_prompt = build_pairwise_prompt(sample_essay_1, sample_essay_2)
call_local_llm([test_prompt], max_new_tokens=64, temperature=0.1)


[{'reasoning': '',
  'preference': 'essay1',
  'raw_text': '{\n  "preference": "essay1"\n}',
  'parse_failed': False}]

In [16]:
def preference_to_label(preference):
    pref = str(preference).lower().strip()
    pref = pref.replace(" ", "")
    pref = pref.replace("_", "")
    pref = pref.replace('"', "")
    pref = pref.replace("'", "")

    if pref in ["essay1", "1"]:
        return 1.0
    if pref in ["essay2", "2"]:
        return 0.0
    if pref in ["tie", "equal", "same", "both"]:
        return 0.5

    return 0.5

def debias_label(label_forward, label_reverse):
    """
    Forward: essay_id_1 as Essay 1, essay_id_2 as Essay 2.
    Reverse: essay_id_2 as Essay 1, essay_id_1 as Essay 2.

    If consistent:
      label_forward == 1 - label_reverse

    Else:
      tie.
    """
    if np.isclose(label_forward, 1.0 - label_reverse):
        return label_forward
    return 0.5

In [17]:
def generate_pairwise_judgments(
    df,
    pairs_df,
    debias=True,
    batch_size=4,
    max_new_tokens=32,
    temperature=0.0,
):
    """Generate pairwise LLM judgments with LCES position-bias correction.

    For Mistral 7B:
    - keep batch_size small
    - use deterministic decoding
    - retry empty generations once
    - convert reverse judgments back to original essay order
    """
    essay_map = df.set_index("essay_id")["essay"].to_dict()
    rows = []

    pair_rows = list(pairs_df.itertuples(index=False))

    def pref_to_original_label(pref, is_reverse=False):
        """
        Convert model preference to label in ORIGINAL pair direction.

        Original pair:
            label =  1 means essay_id_1 is better
            label = -1 means essay_id_2 is better
            label =  0 means tie

        Forward prompt:
            Essay1 = essay_id_1
            Essay2 = essay_id_2

        Reverse prompt:
            Essay1 = essay_id_2
            Essay2 = essay_id_1
        """
        pref = str(pref).strip().lower()

        if pref == "essay1":
            label = 1
        elif pref == "essay2":
            label = -1
        else:
            label = 0

        if is_reverse:
            label = -label

        return label

    def combine_forward_reverse(label_forward, label_reverse_original):
        """
        Conservative debiasing:
        - if forward and reverse agree after converting to original direction, keep label
        - otherwise mark as tie
        """
        if label_forward == label_reverse_original:
            return label_forward
        return 0

    def retry_empty_results(prompts, results):
        """
        Mistral sometimes returns empty raw_text.
        Retry only those failed prompts once.
        """
        failed_indices = [
            i for i, r in enumerate(results)
            if str(r.get("raw_text", "")).strip() == ""
        ]

        if len(failed_indices) == 0:
            return results

        retry_prompts = [prompts[i] for i in failed_indices]

        retry_results = call_local_llm(
            retry_prompts,
            max_new_tokens=max(max_new_tokens, 64),
            temperature=0.0,
        )

        for idx, retry_result in zip(failed_indices, retry_results):
            results[idx] = retry_result

        return results

    for start in tqdm(range(0, len(pair_rows), batch_size)):
        batch = pair_rows[start:start + batch_size]

        forward_prompts = []
        reverse_prompts = []
        meta = []

        for row in batch:
            id1 = int(row.essay_id_1)
            id2 = int(row.essay_id_2)

            essay1 = essay_map[id1]
            essay2 = essay_map[id2]

            forward_prompts.append(build_pairwise_prompt(essay1, essay2))

            if debias:
                reverse_prompts.append(build_pairwise_prompt(essay2, essay1))

            meta.append((id1, id2))

        forward_results = call_local_llm(
            forward_prompts,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
        )

        forward_results = retry_empty_results(forward_prompts, forward_results)

        if debias:
            reverse_results = call_local_llm(
                reverse_prompts,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
            )

            reverse_results = retry_empty_results(reverse_prompts, reverse_results)
        else:
            reverse_results = [None] * len(forward_results)

        for (id1, id2), result_forward, result_reverse in zip(meta, forward_results, reverse_results):
            pref_forward = result_forward.get("preference", "tie")
            label_forward = pref_to_original_label(pref_forward, is_reverse=False)

            raw_forward = result_forward.get("raw_text", "")
            failed_forward = str(raw_forward).strip() == ""

            if debias:
                pref_reverse = result_reverse.get("preference", "tie")
                label_reverse_original = pref_to_original_label(pref_reverse, is_reverse=True)

                raw_reverse = result_reverse.get("raw_text", "")
                failed_reverse = str(raw_reverse).strip() == ""

                final_label = combine_forward_reverse(
                    label_forward,
                    label_reverse_original,
                )

                rows.append({
                    "essay_id_1": id1,
                    "essay_id_2": id2,
                    "label": final_label,

                    "pref_forward": pref_forward,
                    "pref_reverse": pref_reverse,

                    "label_forward_original": label_forward,
                    "label_reverse_original": label_reverse_original,

                    "reason_forward": result_forward.get("reasoning", ""),
                    "reason_reverse": result_reverse.get("reasoning", ""),

                    "raw_forward": raw_forward,
                    "raw_reverse": raw_reverse,

                    "failed_forward": failed_forward,
                    "failed_reverse": failed_reverse,
                })

            else:
                rows.append({
                    "essay_id_1": id1,
                    "essay_id_2": id2,
                    "label": label_forward,

                    "pref_forward": pref_forward,
                    "pref_reverse": None,

                    "label_forward_original": label_forward,
                    "label_reverse_original": None,

                    "reason_forward": result_forward.get("reasoning", ""),
                    "reason_reverse": None,

                    "raw_forward": raw_forward,
                    "raw_reverse": None,

                    "failed_forward": failed_forward,
                    "failed_reverse": None,
                })

        del batch, forward_prompts, reverse_prompts, forward_results, reverse_results, meta

        batch_idx = start // batch_size
        if torch.cuda.is_available() and batch_idx % 10 == 0:
            torch.cuda.empty_cache()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return pd.DataFrame(rows)


def inspect_judgments(judgments):
    print("pref_forward")
    print(judgments["pref_forward"].value_counts(dropna=False))

    if "pref_reverse" in judgments.columns:
        print("\npref_reverse")
        print(judgments["pref_reverse"].value_counts(dropna=False))

    print("\nlabel")
    print(judgments["label"].value_counts(dropna=False))

    tie_rate = (judgments["label"] == 0).mean()
    print(f"\nTie rate: {tie_rate:.2%}")

    if "failed_forward" in judgments.columns:
        print("\nFailure rate:")
        print(judgments[["failed_forward", "failed_reverse"]].mean())

    if tie_rate > 0.60:
        raise ValueError("Tie rate is high. Inspect raw outputs before training RankNet.")


### Pairwise judgment generation
The following cell generates or loads cached LCES pairwise comparisons for the transductive Prompt 1 essay set.


In [18]:
JUDGMENT_CACHE = "all_judgments_p2_mistral7b_paperprompt_m6500_debias.csv"

if os.path.exists(JUDGMENT_CACHE):
    all_judgments = pd.read_csv(JUDGMENT_CACHE)
    print("Loaded cached judgments:", JUDGMENT_CACHE)
else:
    all_judgments = generate_pairwise_judgments(
        df=p1_all,
        pairs_df=all_pairs,
        debias=True,
        batch_size=64,
        max_new_tokens=32,
        temperature=0.0,
    )
    all_judgments.to_csv(JUDGMENT_CACHE, index=False)
    print("Saved judgments:", JUDGMENT_CACHE)

inspect_judgments(all_judgments)

# Free LLM VRAM before loading the embedding model and training RankNet.
if "llm" in globals():
    del llm
if "tokenizer" in globals():
    del tokenizer
if torch.cuda.is_available():
    torch.cuda.empty_cache()


  0%|          | 0/102 [00:00<?, ?it/s]

Saved judgments: all_judgments_p2_mistral7b_paperprompt_m6500_debias.csv
pref_forward
pref_forward
essay1    3799
essay2    2699
tie          2
Name: count, dtype: int64

pref_reverse
pref_reverse
essay1    3839
essay2    2660
tie          1
Name: count, dtype: int64

label
label
-1    2408
 1    2369
 0    1723
Name: count, dtype: int64

Tie rate: 26.51%

Failure rate:
failed_forward    0.0
failed_reverse    0.0
dtype: float64


In [19]:
used_ids = set(all_judgments["essay_id_1"]) | set(all_judgments["essay_id_2"])

print("Total P1 essays:", len(p1_all))
print("Essays appearing in comparisons:", len(used_ids))
print("Coverage:", len(used_ids) / len(p1_all))

pair_count = pd.concat([
    all_judgments["essay_id_1"],
    all_judgments["essay_id_2"],
]).value_counts()

print("\nPair count per essay:")
print(pair_count.describe())


Total P1 essays: 1783
Essays appearing in comparisons: 1783
Coverage: 1.0

Pair count per essay:
count    1783.000000
mean        7.291082
std         1.929150
min         4.000000
25%         6.000000
50%         7.000000
75%         8.000000
max        15.000000
Name: count, dtype: float64


In [20]:
# Paper main experiments use OpenAI text-embedding-3-large. To keep this
# notebook runnable without an API key, we use RoBERTa-base [CLS], which the
# paper also reports as a robust alternative in Appendix B.
EMBEDDING_MODEL = "roberta-base"

embedding_tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL)
embedding_model = AutoModel.from_pretrained(EMBEDDING_MODEL).to(DEVICE)
embedding_model.eval()

@torch.inference_mode()
def encode_essays_cls(df, text_col="essay", batch_size=16, max_length=512):
    all_vecs = []
    texts = df[text_col].tolist()

    for start in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[start:start + batch_size]

        inputs = embedding_tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length,
        ).to(DEVICE)

        outputs = embedding_model(**inputs)
        cls_vec = outputs.last_hidden_state[:, 0, :]
        all_vecs.append(cls_vec.detach().cpu().numpy())

        del inputs, outputs, cls_vec

    return np.vstack(all_vecs).astype(np.float32)


all_embeddings = encode_essays_cls(p1_all, batch_size=16, max_length=512)

print("Embedding model:", EMBEDDING_MODEL)
print("all_embeddings:", all_embeddings.shape)


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  0%|          | 0/112 [00:00<?, ?it/s]

Embedding model: roberta-base
all_embeddings: (1783, 768)


In [21]:
class RankNet(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, dropout=0.3):
        super().__init__()

        self.scorer = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.scorer(x).squeeze(-1)

In [22]:
def train_ranknet(
    df,
    embeddings,
    judgments,
    epochs=100,
    batch_size=4096,
    lr=1e-3,
    weight_decay=0.01,
    hidden_dim=256,
    dropout=0.3,
    device=DEVICE
):
    essay_ids = df["essay_id"].astype(int).tolist()
    id_to_idx = {eid: idx for idx, eid in enumerate(essay_ids)}

    pair_df = judgments[
        judgments["essay_id_1"].isin(id_to_idx) &
        judgments["essay_id_2"].isin(id_to_idx)
    ].copy()

    assert len(pair_df) > 0, "No valid pairwise judgments for this dataframe."

    idx_i = pair_df["essay_id_1"].map(id_to_idx).values
    idx_j = pair_df["essay_id_2"].map(id_to_idx).values
    labels = pair_df["label"].map({
        1: 1.0,
        -1: 0.0,
        0: 0.5,
    }).values.astype(np.float32)
    x = torch.tensor(embeddings, dtype=torch.float32, device=device)

    model = RankNet(
        input_dim=embeddings.shape[1],
        hidden_dim=hidden_dim,
        dropout=dropout
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    loss_fn = nn.BCELoss()

    n = len(pair_df)

    for epoch in range(1, epochs + 1):
        order = np.random.permutation(n)
        epoch_losses = []

        model.train()

        for start in range(0, n, batch_size):
            batch = order[start:start + batch_size]

            bi = torch.tensor(idx_i[batch], dtype=torch.long, device=device)
            bj = torch.tensor(idx_j[batch], dtype=torch.long, device=device)
            y = torch.tensor(labels[batch], dtype=torch.float32, device=device)

            s_i = model(x[bi])
            s_j = model(x[bj])

            pred = torch.sigmoid(s_i - s_j)
            loss = loss_fn(pred, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_losses.append(loss.item())

        if epoch == 1 or epoch % 10 == 0:
            print(f"Epoch {epoch:03d} | loss = {np.mean(epoch_losses):.4f}")

    return model

In [23]:
# Paper hyperparameters: epochs=100, batch_size=4096, hidden_dim=256,
# dropout=0.3, weight_decay=0.01, lr=0.001.
ranknet = train_ranknet(
    df=p1_all,
    embeddings=all_embeddings,
    judgments=all_judgments,
    epochs=100,
    batch_size=4096,
    lr=1e-3,
    weight_decay=0.01,
    hidden_dim=256,
    dropout=0.3,
)


Epoch 001 | loss = 0.6922
Epoch 010 | loss = 0.6435
Epoch 020 | loss = 0.5760
Epoch 030 | loss = 0.5331
Epoch 040 | loss = 0.5089
Epoch 050 | loss = 0.4935
Epoch 060 | loss = 0.4851
Epoch 070 | loss = 0.4768
Epoch 080 | loss = 0.4719
Epoch 090 | loss = 0.4673
Epoch 100 | loss = 0.4645


In [24]:
@torch.no_grad()
def predict_latent_scores(model, embeddings, device=DEVICE):
    model.eval()
    x = torch.tensor(embeddings, dtype=torch.float32, device=device)
    scores = model(x).detach().cpu().numpy()
    return scores


all_latent = predict_latent_scores(ranknet, all_embeddings)

print("latent min:", all_latent.min())
print("latent max:", all_latent.max())
print("latent std:", all_latent.std())
print("latent head:", all_latent[:10])


latent min: -3.79541
latent max: 1.9406983
latent std: 0.95959044
latent head: [ 0.9833658   0.8701128   0.00396819 -0.28324142  0.6392524  -2.2476454
  0.43311703  0.37318313 -1.3259339  -0.45433447]


In [25]:
def fit_latent_mapper(latent_scores_ref, y_min, y_max):
    """Paper's linear latent-score conversion to the rubric range."""
    s_min = float(np.min(latent_scores_ref))
    s_max = float(np.max(latent_scores_ref))

    def mapper(latent_scores):
        latent_scores = np.asarray(latent_scores, dtype=float)

        if np.isclose(s_min, s_max):
            mapped = np.full_like(latent_scores, fill_value=(y_min + y_max) / 2)
        else:
            mapped = (latent_scores - s_min) / (s_max - s_min)
            mapped = mapped * (y_max - y_min) + y_min

        mapped = np.rint(mapped)
        mapped = np.clip(mapped, y_min, y_max)

        return mapped.astype(int)

    return mapper


score_mapper = fit_latent_mapper(all_latent, Y_MIN, Y_MAX)
all_pred = score_mapper(all_latent)

p1_all_scored = p1_all[[
    "essay_id",
    "split",
    "essay_set",
    "essay",
    "domain1_score",
    "domain1_score_norm",
]].copy()

p1_all_scored["latent_score"] = all_latent
p1_all_scored["pred_score"] = all_pred
p1_all_scored["abs_error"] = np.abs(
    p1_all_scored["domain1_score"] - p1_all_scored["pred_score"]
)

print(pd.Series(all_pred).value_counts().sort_index())
p1_all_scored.head()


2       4
3      15
4      60
5     115
6     251
7     346
8     420
9     342
10    189
11     39
12      2
Name: count, dtype: int64


,essay_id,split,essay_set,essay,domain1_score,domain1_score_norm,latent_score,pred_score,abs_error
0,1311,train,1,Dear @CAPS1 @CAPS2 @CAPS3: @CAPS4 people are s...,10.0,0.8,0.983366,10,0.0
1,735,train,1,"As you can see, technology these days is alway...",9.0,0.7,0.870113,10,1.0
2,1152,train,1,"Dear local newspaper, I am writing to you toda...",8.0,0.6,0.003968,9,1.0
3,1314,train,1,"Dear @ORGANIZATION1, Have you ever trully marv...",10.0,0.8,-0.283241,8,2.0
4,1645,train,1,"Dear @ORGANIZATION2, @DATE1, computers are a b...",11.0,0.9,0.639252,10,1.0


In [26]:
metrics_rows = []

for split in ["train", "val", "test", "all"]:
    if split == "all":
        df_eval = p1_all_scored
    else:
        df_eval = p1_all_scored[p1_all_scored["split"] == split]

    metrics_rows.append({
        "split": split,
        **compute_metrics(
            y_true=df_eval["domain1_score"],
            y_pred=df_eval["pred_score"],
        ),
    })

pd.DataFrame(metrics_rows)


,split,QWK,MAE,RMSE,Spearman
0,train,0.523194,1.326122,1.700537,0.610534
1,val,0.539261,1.366667,1.738454,0.653025
2,test,0.461824,1.357746,1.757078,0.558354
3,all,0.513392,1.336511,1.715784,0.604200


In [27]:
p1_test_results = p1_all_scored[p1_all_scored["split"] == "test"].copy()

p1_test_results[[
    "essay_id",
    "domain1_score",
    "pred_score",
    "abs_error",
    "latent_score",
]].head(20)


,essay_id,domain1_score,pred_score,abs_error,latent_score
1428,1723,9.0,7,2.0,-1.021387
1429,1334,8.0,7,1.0,-1.119238
1430,966,7.0,7,0.0,-1.002566
1431,260,7.0,7,0.0,-1.104830
1432,567,8.0,5,3.0,-1.974702
1433,791,8.0,5,3.0,-2.099016
1434,230,9.0,8,1.0,-0.322856
1435,1319,9.0,8,1.0,-0.533325
1436,816,9.0,8,1.0,-0.514643
1437,121,8.0,5,3.0,-2.197624


In [28]:
# Worst predictions on the held-out test split
p1_test_results.sort_values("abs_error", ascending=False)[[
    "essay_id",
    "domain1_score",
    "pred_score",
    "abs_error",
    "latent_score",
    "essay",
]].head(10)


,essay_id,domain1_score,pred_score,abs_error,latent_score,essay
1546,1008,10.0,5,5.0,-1.991589,"Dear editor of the @CAPS1 @CAPS2, @CAPS3 is tr..."
1714,545,2.0,7,5.0,-1.163273,I think that computers are amazing. Computers ...
1720,512,8.0,3,5.0,-3.282884,Dear People Computers are not a really good th...
1517,807,11.0,7,4.0,-0.749679,Computers. An achievement @MONEY1 technology a...
1480,1082,9.0,5,4.0,-1.828365,"Dear @CAPS1, You use a computer a lot right? G..."
1629,1608,8.0,4,4.0,-2.467124,I think that computers are because you can hav...
1684,338,8.0,4,4.0,-2.413032,"Dear, local Newspaper @CAPS1 I read the articl..."
1533,406,8.0,4,4.0,-2.503893,Dear local newspaper I disagree because people...
1486,491,8.0,4,4.0,-2.704961,"Dear, @CAPS1 Who would'nt agree that more and ..."
1676,273,8.0,4,4.0,-2.605460,Dear @CAPS1 @CAPS2 I think that computer have ...


In [29]:
from google.colab import files, runtime
import os
import time

JUDGMENT_CACHE = "all_judgments_p2_mistral7b_paperprompt_m6500_debias.csv"

possible_paths = [
    f"/content/{JUDGMENT_CACHE}",
    f"/content/output/{JUDGMENT_CACHE}",
]

SAVE_PATH = None
for path in possible_paths:
    if os.path.isfile(path):
        SAVE_PATH = path
        break

if SAVE_PATH is None:
    raise FileNotFoundError(
        "Không tìm thấy file ở:\n" + "\n".join(possible_paths)
    )

print(f"Đang tải: {SAVE_PATH}")
files.download(SAVE_PATH)



Đang tải: /content/all_judgments_p2_mistral7b_paperprompt_m6500_debias.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>